# 07 · Tran–Vu characterization

**Grand Challenge Labs · Coupling-Phase Spectroscopy**

Characterize when the moderate-gap refinement is valid, useful, and genuinely sharper than classical Davis–Kahan.


## Release contract

| Contract | Declared value |
|---|---|
| **Scientific question** | Which combinations of gap, perturbation size, directional coupling, cluster breadth, complexity, and non-normality make the Tran–Vu singular-space certificate useful? |
| **Default path** | Run the governed fixture matrix and full deterministic parameter sweeps on CPU. |
| **Evidence boundary** | This characterizes a synthetic certificate. It does not establish predictive or causal value for model training. |
| **Primary outputs** | CSV/JSON datasets, nine PNG/SVG figure pairs, a standalone HTML report, acceptance record, manifest, and `/content/cps-export.zip`. |


## Interpretation checklist

- [ ] Confirm that all ten governed fixtures match their declared outcomes.
- [ ] Distinguish theorem applicability from an informative or sharper bound.
- [ ] Read the directional-coupling and admission-map plots before treating a small gap as dangerous.
- [ ] Keep singular-space stability separate from eigenvector conditioning, pseudospectral growth, and projection closure.
- [ ] Verify that all nine PNG/SVG figure pairs and the HTML report are present before export.


## Learning objectives

By the end of the run, you should be able to identify the moderate-gap lower and upper boundaries, explain why weak local coupling can improve a global-norm bound, see the quadratic halving-rank penalty, and state precisely what the certificate does not control.


In [ ]:
import importlib
import os
import pathlib
import subprocess
import sys

REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")

print("[BOOT] Preparing the CPS repository", flush=True)
print(f"[BOOT] source={REPO_URL}", flush=True)
print(f"[BOOT] ref={GIT_REF}", flush=True)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", GIT_REF], check=True)
os.chdir(repo)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-e", ".[dev,notebooks]"
], check=True)
src_dir = repo / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
importlib.invalidate_caches()
import cps
print(f"[BOOT] CPS import verified from {cps.__file__}", flush=True)


In [ ]:
from cps.notebook import apply_release_theme, stage_banner
stage_banner(
    "0",
    "Record the runtime contract",
    objective="Expose the software environment and source revision before execution.",
    deliverable="A visible runtime inventory for reproducibility.",
)
from cps.notebook import show_environment
apply_release_theme()
runtime = show_environment()


## Stage 1 — verify the certificate and fixture contracts

The focused test slice checks theorem gates, complex realification, rank handling, and the declared outcomes of the governed regime matrix.


In [ ]:
from cps.notebook import stage_banner
stage_banner(
    "1",
    "Verify the certificate contracts",
    objective="Reject implementation or fixture drift before generating characterization evidence.",
    deliverable="A passing focused pytest record.",
)
command = [
    sys.executable, "-m", "pytest", "-vv",
    "tests/test_subspace_stability.py",
    "tests/test_tran_vu_experiment_package.py::test_governed_regime_matrix_matches_declared_outcomes",
]
print("[TEST]", " ".join(command), flush=True)
subprocess.run(command, check=True)


## Stage 2 — execute the complete characterization

The runner emits governed tables, mechanism sweeps, the two-dimensional admission map, non-normality separation, realification validation, and a standalone visual report.


In [ ]:
from cps.notebook import stage_banner
stage_banner(
    "2",
    "Run the full characterization",
    objective="Map the theorem's useful and invalid regimes with deterministic synthetic operators.",
    deliverable="A complete packet under /content/cps-artifacts/tran-vu-characterization.",
)
from experiments.tran_vu.run import run_characterization

output_dir = pathlib.Path("/content/cps-artifacts/tran-vu-characterization")
acceptance = run_characterization(output_dir)
assert acceptance["passed"], acceptance
print(f"[CHARACTERIZATION] report={output_dir / 'index.html'}", flush=True)


## Stage 3 — inspect the evidence visually

The table gives the governed regime outcomes. The figures then show the directional-coupling crossover, theorem boundaries, halving-rank penalty, admission region, bound coverage, non-normality separation, and realification error.


In [ ]:
from cps.notebook import stage_banner
stage_banner(
    "3",
    "Display the governed report",
    objective="Make every acceptance decision and visual mechanism available for direct inspection.",
    deliverable="Visible tables, acceptance gates, and all nine plots.",
)
import json
from IPython.display import HTML, Image, Markdown, display

summary = json.loads((output_dir / "report.json").read_text(encoding="utf-8"))
summary_lines = [
    f"**Acceptance:** `{summary['acceptance_passed']}`",
    f"**Regimes:** `{summary['regime_count']}`",
    f"**Applicable:** `{summary['theorem_applicable_count']}`",
    f"**Admitted:** `{summary['admitted_count']}`",
    f"**Figures:** `{summary['figure_count']}`",
]
display(Markdown("  \n".join(summary_lines)))
for figure in sorted((output_dir / "figures").glob("*.png")):
    display(Markdown(f"### {figure.stem.replace('_', ' ').title()}"))
    display(Image(filename=str(figure)))
display(HTML((output_dir / "index.html").read_text(encoding="utf-8")))


## Final stage — export the evidence packet

The common CPS export step packages the complete artifact tree for Colab CLI retrieval or manual preservation.


In [ ]:
from cps.notebook import export_artifacts, stage_banner
stage_banner(
    "EXPORT",
    "Package the evidence",
    objective="Collect datasets, plots, acceptance records, report, and manifest into one archive.",
    deliverable="/content/cps-export.zip",
)
archive = export_artifacts()
print(f"[EXPORT] archive={archive}", flush=True)
